# Credit Card Fraud Detection
Based on: [Credit Fraud — Dealing with Imbalanced Datasets](https://www.kaggle.com/code/janiobachmann/credit-fraud-dealing-with-imbalanced-datasets)

## 1. Problem Framing

**Goal:** Predict whether a credit card transaction is fraudulent (`Class = 1`) or legitimate (`Class = 0`).

**ML Task:** Binary classification with severe class imbalance (~0.17% fraud).

**Key Rules (from reference notebook):**
- Never test on oversampled/undersampled data — always test on the original holdout
- If using cross-validation, apply resampling *during* CV, not before
- Don't use accuracy as the primary metric — use F1, Precision/Recall, ROC-AUC

**Business Impact:** Minimize missed fraud (false negatives) while keeping false positives low to avoid blocking legitimate transactions.

## 2. Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Preprocessing & evaluation
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    accuracy_score, classification_report, confusion_matrix,
    precision_recall_curve, average_precision_score
)
from collections import Counter

# Imbalanced learning
from imblearn.pipeline import make_pipeline as imbalanced_make_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss
from imblearn.metrics import classification_report_imbalanced

# Dimensionality reduction
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA, TruncatedSVD

# Load data
df = pd.read_csv('/Users/jheeldoshi/Documents/GitHub/Claude-Analytics-Engineering-Data-Setup/data/creditcard.csv')
print(f'Shape: {df.shape}')
df.head()

## 3. Data Audit

## Dataset Context

### Schema Report

| Column | Inferred Meaning | Feature Type | Include/Exclude | Notes |
|---|---|---|---|---|
| Time | Seconds since first transaction | datetime-like | Include (scaled) | Treat as ordering signal, not raw datetime |
| V1–V28 | PCA-transformed anonymized features | numeric | Include | Already scaled via PCA — no further feature engineering possible |
| Amount | Transaction amount (USD) | numeric | Include (scaled) | High skew; max $25,691 vs mean $88 — needs RobustScaler |
| Class | Fraud label | target | Target | 0 = legitimate, 1 = fraud |

### Modeling Risks

| Risk | Detail |
|---|---|
| Severe class imbalance | 99.83% legit / 0.17% fraud — accuracy is misleading |
| No identifiers to exclude | All columns are features or target |
| Time ordering | `Time` may require time-based splitting depending on business need |
| Amount skew | Needs scaling before modeling |
| No raw features | V1–V28 are PCA-transformed — interpret correlations, not raw meaning |

### Recommended Preprocessing

- Scale `Time` and `Amount` using `RobustScaler`
- No encoding needed (all numeric)
- No missing values
- Address class imbalance via random undersampling or SMOTE

### Validation Strategy

**Stratified split** — preserves the 0.17% fraud rate in both train and test sets.

### Recommended Models & Metrics

| | Recommendation |
|---|---|
| Baseline | Logistic Regression (`class_weight="balanced"`) |
| Candidates | Random Forest, Gradient Boosting |
| Primary Metrics | ROC-AUC, Precision-Recall AUC, F1 (fraud class) |
| Avoid | Accuracy as primary metric |

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print()
print('Class distribution:')
fraud_pct = (df['Class'].value_counts() / len(df) * 100).round(2)
print(f'No Frauds {fraud_pct[0]} % of the dataset')
print(f'Frauds    {fraud_pct[1]} % of the dataset')
print()
df.describe()

In [ ]:
# Class distribution plot
colors = ['#2ECC71', '#E74C3C']
fig, ax = plt.subplots(figsize=(6, 4))
df['Class'].value_counts().plot(kind='bar', color=colors, ax=ax)
ax.set_title('Class Distributions \n (0: No Fraud || 1: Fraud)', fontsize=14)
ax.set_xticklabels(['No Fraud', 'Fraud'], rotation=0)
plt.tight_layout()
plt.show()

## 4. Preprocessing — Scaling and Distributing

In [ ]:
# Scale Time and Amount using RobustScaler (less sensitive to outliers)
rob_scaler = RobustScaler()
df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time']   = rob_scaler.fit_transform(df['Time'].values.reshape(-1, 1))

# Drop original unscaled columns
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# Move scaled columns to front
scaled_amount = df['scaled_amount']
scaled_time   = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

print('New columns:', list(df.columns[:5]), '...')
df.head()

## 5. Splitting the Data (Original DataFrame)

**Important:** Split the original data *before* any resampling. We will always test on this original test set.

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

# Stratified split to preserve fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'Train fraud rate: {y_train.mean():.4%}')
print(f'Test fraud rate:  {y_test.mean():.4%}')

# StratifiedKFold for cross-validation
sss = StratifiedKFold(n_splits=5, random_state=None, shuffle=False)
for train_idx, test_idx in sss.split(X, y):
    print(f'Train: {train_idx[:3]}... Test: {test_idx[:3]}...')

## 6. Random Under-Sampling (50/50 Subsample)

In [ ]:
# Create balanced subsample: 492 fraud + 492 non-fraud
fraud_df     = df[df['Class'] == 1]
non_fraud_df = df[df['Class'] == 0].sample(len(fraud_df), random_state=42)

sub_sample = pd.concat([fraud_df, non_fraud_df]).sample(frac=1, random_state=42).reset_index(drop=True)

X_sub = sub_sample.drop('Class', axis=1)
y_sub = sub_sample['Class']

print('Subsample class distribution:')
print(y_sub.value_counts(normalize=True))
print(f'Total rows: {len(sub_sample)}')

## 7. Exploratory Data Analysis

In [ ]:
# Correlation matrix on subsample (not original — imbalance distorts correlations)
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

corr_original = df.corr()
sns.heatmap(corr_original, cmap='coolwarm_r', ax=ax1)
ax1.set_title('Imbalanced Correlation Matrix \n(Original DataFrame)', fontsize=14)

corr_sub = sub_sample.corr()
sns.heatmap(corr_sub, cmap='coolwarm_r', ax=ax2)
ax2.set_title('Equal Ratio Correlation Matrix \n(Balanced Subsample)', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Negative correlations with fraud: V17, V14, V12, V10
# Positive correlations with fraud: V2, V4, V11, V19
f, axes = plt.subplots(2, 4, figsize=(20, 10))

neg_corr = ['V17', 'V14', 'V12', 'V10']
pos_corr = ['V2', 'V4', 'V11', 'V19']

for i, feat in enumerate(neg_corr):
    sns.boxplot(x='Class', y=feat, data=sub_sample, ax=axes[0, i], palette=['#2ECC71', '#E74C3C'])
    axes[0, i].set_title(f'{feat} vs Class (Negative Corr)')

for i, feat in enumerate(pos_corr):
    sns.boxplot(x='Class', y=feat, data=sub_sample, ax=axes[1, i], palette=['#2ECC71', '#E74C3C'])
    axes[1, i].set_title(f'{feat} vs Class (Positive Corr)')

plt.tight_layout()
plt.show()

## 8. Anomaly Detection — Outlier Removal (IQR Method)

We remove **extreme outliers** from the most correlated features (V14, V12, V10) on the fraud cases in the subsample.

**Tradeoff:** Lower IQR multiplier → more outliers removed → risk of information loss. We use 1.5x IQR.

### Outlier Detection — Before Removal

Check the distribution and count of outliers per feature before deciding to remove them.

In [ ]:
# Distributions of high-correlation features on fraud cases
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, ["V14", "V12", "V10"]):
    fraud_vals = sub_sample[feat][sub_sample["Class"] == 1]
    ax.hist(fraud_vals, bins=40, color="#E74C3C", alpha=0.7)
    ax.set_title(f"{feat} — Fraud Distribution")
    ax.set_xlabel(feat)
plt.suptitle("Feature Distributions (Fraud cases) — Before Outlier Removal", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# IQR outlier counts per feature (fraud cases only) — detection only, no removal yet
rows = []
for feat in ["V14", "V12", "V10"]:
    fraud_vals = sub_sample[feat][sub_sample["Class"] == 1]
    q25, q75 = fraud_vals.quantile(0.25), fraud_vals.quantile(0.75)
    iqr = q75 - q25
    lower, upper = q25 - 1.5 * iqr, q75 + 1.5 * iqr
    outliers = fraud_vals[(fraud_vals < lower) | (fraud_vals > upper)]
    rows.append({"Feature": feat, "Q25": round(q25,2), "Q75": round(q75,2),
                 "IQR": round(iqr,2), "Lower": round(lower,2), "Upper": round(upper,2),
                 "Outlier Count": len(outliers), "Outlier Values": outliers.values.tolist()})

outlier_df = pd.DataFrame(rows).set_index("Feature")
print(f"Rows before removal: {len(sub_sample)}")
outlier_df

In [ ]:
def remove_outliers(df, feature):
    fraud_feat = df[feature][df['Class'] == 1]
    q25, q75   = fraud_feat.quantile(0.25), fraud_feat.quantile(0.75)
    iqr        = q75 - q25
    cut_off    = iqr * 1.5
    lower, upper = q25 - cut_off, q75 + cut_off

    outliers = fraud_feat[(fraud_feat < lower) | (fraud_feat > upper)]
    print(f'{feature} | Q25: {q25:.2f} | Q75: {q75:.2f} | IQR: {iqr:.2f} | Lower: {lower:.2f} | Upper: {upper:.2f}')
    print(f'Outliers found: {len(outliers)}')

    df = df.drop(df[(df[feature] < lower) | (df[feature] > upper)].index)
    print(f'Rows after removal: {len(df)}')
    print('-' * 80)
    return df

new_df = sub_sample.copy()
for feat in ['V14', 'V12', 'V10']:
    new_df = remove_outliers(new_df, feat)

print(f'\nFinal subsample size: {len(new_df)}')

In [ ]:
# Visualize outlier reduction
f, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, feat in enumerate(['V14', 'V12', 'V10']):
    sns.boxplot(x='Class', y=feat, data=new_df, ax=axes[i], palette=['#2ECC71', '#E74C3C'])
    axes[i].set_title(f'{feat} — After Outlier Removal')
plt.tight_layout()
plt.show()

## 9. Dimensionality Reduction & Clustering (t-SNE, PCA, TruncatedSVD)

In [ ]:
X_sub_clean = new_df.drop('Class', axis=1)
y_sub_clean = new_df['Class']

# t-SNE
t0 = time.time()
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_sub_clean)
print(f't-SNE took {time.time() - t0:.1f}s')

# PCA
t0 = time.time()
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_sub_clean)
print(f'PCA took {time.time() - t0:.3f}s')

# TruncatedSVD
t0 = time.time()
svd = TruncatedSVD(n_components=2, random_state=42)
X_svd = svd.fit_transform(X_sub_clean)
print(f'TruncatedSVD took {time.time() - t0:.4f}s')

In [ ]:
f, axes = plt.subplots(1, 3, figsize=(22, 6))
blue = mpatches.Patch(color='#0A0AFF', label='No Fraud')
red  = mpatches.Patch(color='#AF0000', label='Fraud')

for ax, X_2d, title in zip(axes,
                            [X_tsne, X_pca, X_svd],
                            ['t-SNE', 'PCA', 'TruncatedSVD']):
    ax.scatter(X_2d[y_sub_clean==0, 0], X_2d[y_sub_clean==0, 1],
               c='#0A0AFF', alpha=0.5, s=10, label='No Fraud')
    ax.scatter(X_2d[y_sub_clean==1, 0], X_2d[y_sub_clean==1, 1],
               c='#AF0000', alpha=0.5, s=10, label='Fraud')
    ax.set_title(title, fontsize=13)
    ax.legend(handles=[blue, red])

plt.suptitle('Dimensionality Reduction — Fraud vs No Fraud Clustering', fontsize=15)
plt.tight_layout()
plt.show()

## 10. Classifiers — Random UnderSampling

Train on the balanced subsample, test on the **original** test set.

In [ ]:
X_sub_train, X_sub_test, y_sub_train, y_sub_test = train_test_split(
    X_sub_clean, y_sub_clean, test_size=0.2, random_state=42
)

classifiers = {
    'Logistic Regression':   LogisticRegression(max_iter=1000),
    'KNeighborsClassifier':  KNeighborsClassifier(),
    'SVC':                   SVC(),
    'DecisionTreeClassifier': DecisionTreeClassifier()
}

for name, clf in classifiers.items():
    clf.fit(X_sub_train, y_sub_train)
    train_score = clf.score(X_sub_train, y_sub_train)
    print(f'{name}: training accuracy = {train_score * 100:.0f}%')

In [ ]:
# Cross-validation scores on subsample
for name, clf in classifiers.items():
    cv_scores = cross_val_score(clf, X_sub_clean, y_sub_clean, cv=5)
    print(f'{name} CV Score: {cv_scores.mean() * 100:.2f}%')

## 11. GridSearchCV — Logistic Regression Tuning

In [ ]:
# Hyperparameter search
log_reg_params = {'penalty': ['l1', 'l2'], 'C': [0.001, 0.01, 0.1, 1, 10, 100]}
knear_params   = {'n_neighbors': [2, 5, 7, 11], 'weights': ['uniform', 'distance']}
svc_params     = {'C': [0.5, 1.0, 1.5], 'kernel': ['rbf', 'linear']}
tree_params    = {'max_depth': [2, 4, 6, 8, 10], 'criterion': ['gini', 'entropy']}

grid_log  = GridSearchCV(LogisticRegression(max_iter=1000, solver='liblinear'), log_reg_params)
grid_knear = GridSearchCV(KNeighborsClassifier(), knear_params)
grid_svc  = GridSearchCV(SVC(), svc_params)
grid_tree = GridSearchCV(DecisionTreeClassifier(), tree_params)

for name, grid in [('LogReg', grid_log), ('KNN', grid_knear), ('SVC', grid_svc), ('DecisionTree', grid_tree)]:
    grid.fit(X_sub_train, y_sub_train)
    print(f'{name} best params: {grid.best_params_}')

best_log  = grid_log.best_estimator_
best_knear = grid_knear.best_estimator_
best_svc  = grid_svc.best_estimator_
best_tree = grid_tree.best_estimator_

## 12. Precision-Recall — Logistic Regression Deeper Analysis

In [ ]:
log_reg_pred   = cross_val_score(best_log, X_sub_clean, y_sub_clean, cv=5, scoring='recall')
knear_pred     = cross_val_score(best_knear, X_sub_clean, y_sub_clean, cv=5, scoring='recall')
svc_pred       = cross_val_score(best_svc, X_sub_clean, y_sub_clean, cv=5, scoring='recall')
tree_pred      = cross_val_score(best_tree, X_sub_clean, y_sub_clean, cv=5, scoring='recall')

print(f'Logistic Regression Recall:   {log_reg_pred.mean():.4f}')
print(f'KNears Neighbors Recall:      {knear_pred.mean():.4f}')
print(f'Support Vector Classifier:    {svc_pred.mean():.4f}')
print(f'Decision Tree Recall:         {tree_pred.mean():.4f}')

In [ ]:
# Evaluate on original test set
print('=' * 70)
for name, clf in [('Logistic Regression', best_log), ('KNN', best_knear),
                  ('SVC', best_svc), ('Decision Tree', best_tree)]:
    clf.fit(X_sub_train, y_sub_train)
    y_pred = clf.predict(X_test)
    print(f'\n{name}:')
    print(classification_report(y_test, y_pred, target_names=['No Fraud', 'Fraud']))
    print(f'ROC-AUC: {roc_auc_score(y_test, y_pred):.4f}')
    print('-' * 70)

In [ ]:
# Confusion matrices
f, axes = plt.subplots(1, 4, figsize=(22, 5))
labels  = [('Logistic Regression', best_log), ('KNN', best_knear),
           ('SVC', best_svc), ('Decision Tree', best_tree)]

for ax, (name, clf) in zip(axes, labels):
    y_pred = clf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Fraud', 'Fraud'],
                yticklabels=['No Fraud', 'Fraud'])
    ax.set_title(name)

plt.suptitle('Confusion Matrices — Tested on Original Test Set', fontsize=14)
plt.tight_layout()
plt.show()

## 13. NearMiss Under-Sampling

In [ ]:
nm = NearMiss()
X_nm, y_nm = nm.fit_resample(X_train, y_train)
print(f'NearMiss label distribution: {Counter(y_nm)}')

log_reg_nm = LogisticRegression(max_iter=1000)
log_reg_nm.fit(X_nm, y_nm)
y_pred_nm = log_reg_nm.predict(X_test)

print(classification_report_imbalanced(y_test, y_pred_nm))
print(f'ROC-AUC (NearMiss): {roc_auc_score(y_test, y_pred_nm):.4f}')

## 14. SMOTE Over-Sampling

**Correct approach:** Apply SMOTE *inside* the cross-validation loop using `imblearn.pipeline` to avoid data leakage.

In [ ]:
print(f'Length of X_train: {len(X_train)} | y_train: {len(y_train)}')
print(f'Length of X_test:  {len(X_test)}  | y_test:  {len(y_test)}')

# SMOTE pipeline — resampling occurs DURING CV, not before
smote_pipeline = imbalanced_make_pipeline(
    SMOTE(sampling_strategy='minority', random_state=42),
    LogisticRegression(max_iter=1000)
)

smote_model = smote_pipeline.fit(X_train, y_train)
smote_pred  = smote_model.predict(X_test)

print(f'\nSMOTE + Logistic Regression (tested on original test set):')
print(classification_report(y_test, smote_pred, target_names=['No Fraud', 'Fraud']))
print(f'ROC-AUC: {roc_auc_score(y_test, smote_pred):.4f}')

In [ ]:
# Precision-Recall curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Undersample
best_log.fit(X_sub_train, y_sub_train)
y_score_under = best_log.predict_proba(X_test)[:, 1] if hasattr(best_log, 'predict_proba') else best_log.decision_function(X_test)
prec_u, rec_u, _ = precision_recall_curve(y_test, y_score_under)
ap_u = average_precision_score(y_test, y_score_under)
axes[0].plot(rec_u, prec_u)
axes[0].set_title(f'UnderSampling PR Curve\nAvg Precision = {ap_u:.2f}')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')

# Oversample
smote_probs   = smote_model.predict_proba(X_test)[:, 1]
prec_o, rec_o, _ = precision_recall_curve(y_test, smote_probs)
ap_o = average_precision_score(y_test, smote_probs)
axes[1].plot(rec_o, prec_o, color='orange')
axes[1].set_title(f'SMOTE Oversampling PR Curve\nAvg Precision = {ap_o:.2f}')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')

plt.tight_layout()
plt.show()

## 15. Neural Network — Keras

Compare performance of a simple NN trained on undersampled vs SMOTE-oversampled data.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

def build_model(n_features):
    model = Sequential([
        Dense(n_features, input_dim=n_features, activation='relu'),
        Dense(32, activation='relu'),
        Dense(2, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

n_features = X_train.shape[1]
build_model(n_features).summary()

In [ ]:
# Train on undersampled data
model_under = build_model(n_features)
model_under.fit(X_sub_clean, y_sub_clean,
                validation_split=0.2, epochs=20, batch_size=32, verbose=1)

y_pred_under_nn = model_under.predict(X_test).argmax(axis=1)

print('\nNN (UnderSampling) — Confusion Matrix:')
cm_under = confusion_matrix(y_test, y_pred_under_nn)
print(cm_under)

In [ ]:
# Train on SMOTE-oversampled data
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_smote, y_smote = sm.fit_resample(X_train, y_train)
print(f'SMOTE resampled size: {Counter(y_smote)}')

model_over = build_model(n_features)
model_over.fit(X_smote, y_smote,
               validation_split=0.2, epochs=20, batch_size=300, verbose=1)

y_pred_over_nn = model_over.predict(X_test).argmax(axis=1)

print('\nNN (SMOTE Oversampling) — Confusion Matrix:')
cm_over = confusion_matrix(y_test, y_pred_over_nn)
print(cm_over)

In [ ]:
# Side-by-side confusion matrices
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_under, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['No Fraud', 'Fraud'], yticklabels=['No Fraud', 'Fraud'])
ax1.set_title('NN — UnderSampling')

sns.heatmap(cm_over, annot=True, fmt='d', cmap='Blues', ax=ax2,
            xticklabels=['No Fraud', 'Fraud'], yticklabels=['No Fraud', 'Fraud'])
ax2.set_title('NN — SMOTE Oversampling')

plt.suptitle('Neural Network: UnderSampling vs Oversampling', fontsize=14)
plt.tight_layout()
plt.show()

## 16. Model Comparison Summary

In [ ]:
results = []

for name, clf in [('LogReg (UnderSample)', best_log),
                  ('KNN (UnderSample)', best_knear),
                  ('SVC (UnderSample)', best_svc),
                  ('DecisionTree (UnderSample)', best_tree),
                  ('LogReg (NearMiss)', log_reg_nm)]:
    y_p = clf.predict(X_test)
    results.append({
        'Model': name,
        'ROC-AUC': roc_auc_score(y_test, y_p),
        'Recall (Fraud)': recall_score(y_test, y_p),
        'Precision (Fraud)': precision_score(y_test, y_p),
        'F1 (Fraud)': f1_score(y_test, y_p)
    })

for name, y_p in [('SMOTE + LogReg', smote_pred),
                  ('NN (UnderSample)', y_pred_under_nn),
                  ('NN (SMOTE)', y_pred_over_nn)]:
    results.append({
        'Model': name,
        'ROC-AUC': roc_auc_score(y_test, y_p),
        'Recall (Fraud)': recall_score(y_test, y_p),
        'Precision (Fraud)': precision_score(y_test, y_p),
        'F1 (Fraud)': f1_score(y_test, y_p)
    })

pd.DataFrame(results).set_index('Model').sort_values('ROC-AUC', ascending=False).round(4)

## 17. Final Recommendation

**Best Model:**
- *(To be filled after running Section 16)*

**Key Takeaways:**
- *(To be filled after running the notebook)*

**Business Recommendation:**
- *(To be filled after reviewing results — consider false negative vs false positive cost tradeoff)*